In [17]:
import os


In [18]:
import pandas as pd

In [19]:
df_path = "/scratch/yag1/ego4d_data/v2/queries_with_talk.csv"

query_df = pd.read_csv(df_path)

In [20]:
# !pip install -U google-genai

In [21]:
from google import genai
from dotenv import load_dotenv
from google.genai import types

In [22]:
env_path = os.getcwd() + "/.env"
load_dotenv(env_path)

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [23]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [24]:
video_system_prompt = """
You are a precise text-transformation tool for multi-modal video retrieval. Your job is to convert an interrogative query into a highly specific declarative format by extracting the core action and its objects/locations, without adding ANY unmentioned information.

### Step 1: Extract the Core Action Phrase
Convert the main action of the question into a neutral continuous phrase (e.g., "talking to the mechanic", "putting down the keys", "opening the refrigerator"). Do not add outside descriptions or guess ambient details.

### Step 2: Apply Modality Prefixes
- VideoPrism Format: "A video of a person [Core Action Phrase]."

Respond strictly in this format:
{
  "videoprism": "VideoPrism text here"
}

### Examples:

Query: "When did I talk to the mechanic?"
Output:
{
  "videoprism": "A video of a person talking to the mechanic."
}

Query: "Where did I put the keys?"
Output:
{
  "videoprism": "A video of a person putting down the keys."
}

Query: "What did I take out of the refrigerator?"
Output:
{
  "videoprism": "A video of a person taking something out of the refrigerator."
}
"""

In [25]:
video_config = types.GenerateContentConfig(
    system_instruction=video_system_prompt,
    max_output_tokens=2048
)

In [26]:
audio_system_prompt = """
You are a precise text-transformation tool for multi-modal audio retrieval. Your job is to convert an interrogative query into a highly specific declarative format by extracting the core action and its objects/locations, without adding ANY unmentioned information.

### Step 1: Extract the Core Action Phrase
Convert the main action of the question into a neutral continuous phrase (e.g., "talking to the mechanic", "putting down the keys", "opening the refrigerator"). Do not add outside descriptions or guess ambient details.

### Step 2: Apply Modality Prefixes
- CLAP Format: "The sound of a person [Core Action Phrase]."

Respond strictly in this format:
{
  "clap": "CLAP text here",
}

### Examples:

Query: "When did I talk to the mechanic?"
Output:
{
  "clap": "The sound of a person talking to the mechanic.",
}

Query: "Where did I put the keys?"
Output:
{
  "clap": "The sound of a person putting down the keys.",
}

Query: "What did I take out of the refrigerator?"
Output:
{
  "clap": "The sound of a person taking something out of the refrigerator.",
}
"""

In [27]:
audio_config = types.GenerateContentConfig(
    system_instruction=audio_system_prompt,
    max_output_tokens=2048
)

In [28]:
MODEL="gemini-3.1-pro-preview"
# MODEL="gemini-3.1-flash-lite"

In [29]:
import ast

In [30]:
def reformulate_audio_query(text):
    
    response = client.models.generate_content(
    model=MODEL, 
    contents=[text],
    config=audio_config
    )
    
    new_queries = response.text
    
    if "```" in new_queries:
        new_queries = new_queries.replace("```", "")
    if "json" in new_queries:
        new_queries = new_queries.replace("json", "")
    new_queries = new_queries.strip()
    
    new_queries = ast.literal_eval(new_queries)
    
    return new_queries["clap"]

In [31]:
query_df["clap_query"] = query_df.apply(
    lambda row: reformulate_audio_query(row["query"]),
    axis=1)

In [32]:
def reformulate_video_query(text):
    response = client.models.generate_content(
    model=MODEL, 
    contents=[text],
    config=video_config
    )
    
    new_queries = response.text
    
    if "```" in new_queries:
        new_queries = new_queries.replace("```", "")
    if "json" in new_queries:
        new_queries = new_queries.replace("json", "")
    new_queries = new_queries.strip()
    
    new_queries = ast.literal_eval(new_queries)
    
    return new_queries["videoprism"]


In [33]:
query_df["videoprism_query"] = query_df.apply(
    lambda row: reformulate_video_query(row["query"]),
    axis=1)

In [34]:
query_df.head()

,clip_id,query,query_start_sec,query_end_sec,query_duration,clap_query,videoprism_query
0,a325ce85-cae5-4faa-99bb-7272918fcf19,when did i talk to the cashier at the store,403.38680,408.57537,5.18857,The sound of a person talking to the cashier a...,A video of a person talking to the cashier at ...
1,a325ce85-cae5-4faa-99bb-7272918fcf19,who did i talk to in the supermarket,404.36400,405.51063,1.14663,The sound of a person talking to someone in th...,A video of a person talking to someone in the ...
2,b69e6150-0309-4202-bf4a-9a342f80d6d7,who did i talk to in the kitchen?,84.48200,88.18600,3.70400,The sound of a person talking to someone in th...,A video of a person talking to someone in the ...
3,35cd9ace-642f-4550-8e63-a5c2caae89ed,who did i talk to in the workshop?,176.98861,180.98800,3.99939,The sound of a person talking to someone in th...,A video of a person talking to someone in the ...
4,35cd9ace-642f-4550-8e63-a5c2caae89ed,who did i talk to in the workshop?,301.55537,323.55500,21.99963,The sound of a person talking to someone in th...,A video of a person talking to someone in the ...


In [35]:
query_df.to_csv("/scratch/yag1/ego4d_data/v2/updated_queries_with_talk.csv", index=False)